# 01 - Preprocessing check

**What this notebook does**: proves the input pipeline is doing what the models assume, before any
CPU time is spent training. Cheap to run, and it rules out an entire category of "the model won't
learn" dead ends.

**What must already exist**: the split CSVs from notebook 00.

**The four things being verified**
1. Images come out at 224x224x3, `float32`, in the raw **[0, 255]** range - rescaling happens
   *inside* the model, so a model can never be paired with the wrong input scaling.
2. Labels stay aligned with the dataframe order for val/test (unshuffled), which is what makes
   `predict()` comparable to `y_true` at all.
3. One-hot targets are produced when asked for - MiniConvNet trains with label smoothing (LESSON 2),
   which requires them.
4. Augmentation is visible and conservative: CT slices are not vertically flipped or heavily rotated.

**Cost**: no training. A couple of minutes, mostly image decoding.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('IMG_SIZE:', IMG_SIZE, '| BATCH_SIZE:', BATCH_SIZE)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.data_utils import (load_split, make_dataset, make_split_datasets,
                            split_counts, labels_of, assert_no_leakage)
from src.train_utils import set_global_seeds, compute_report

set_global_seeds(SEED)
for k, v in compute_report().items():
    print(f'{k}: {v}')

## 1. Load both splits and re-check leakage

Re-asserting here (not just in notebook 00) catches the case where a split CSV was edited or
regenerated by hand between notebooks.

**Looks right**: `faithful` counts matching notebook 00, `clean` smaller, and the clean assertion
passing.

In [ ]:
faithful = load_split('faithful')
clean = load_split('clean')

for name, sdf in (('faithful', faithful), ('clean', clean)):
    print(f'--- {name} ---')
    print(split_counts(sdf).to_string())
    print()

assert_no_leakage(clean)
print('clean split no-leakage assertion: PASSED')

spans = faithful.groupby('hash')['split'].nunique()
print(f'faithful: {int((spans > 1).sum())} content hashes span more than one split '
      '(expected > 0 - this is the documented, deliberate leakage of the paper-comparable split)')

## 2. Batch geometry, dtype and value range

**Looks right**: `(32, 224, 224, 3)`, `float32`, min ~0 and max ~255. If the max is ~1.0 the data is
being rescaled twice (once here and once inside the model) and every model will underperform for a
reason that has nothing to do with the architecture.

In [ ]:
train_ds, val_ds, test_ds, frames = make_split_datasets(faithful)

x, y = next(iter(train_ds))
print('image batch:', x.shape, x.dtype)
print('label batch:', y.shape, y.dtype, '(sparse integer labels)')
print(f'pixel range: min={float(tf.reduce_min(x)):.2f} max={float(tf.reduce_max(x)):.2f} '
      f'mean={float(tf.reduce_mean(x)):.2f}')
print()
assert x.shape[1:] == IMG_SHAPE, 'unexpected image shape'
assert float(tf.reduce_max(x)) > 1.5, 'data looks pre-rescaled - the model rescales, not the pipeline'
print('geometry/dtype/range checks: PASSED')

## 3. One-hot targets for label smoothing (LESSON 2)

MiniConvNet is compiled with `CategoricalCrossentropy(label_smoothing=0.05)`, which has no sparse
counterpart, so its datasets must be built with `one_hot=True`. This cell confirms both forms exist
and agree.

**Looks right**: one-hot rows of width 4 summing to 1, whose argmax equals the sparse label.

In [ ]:
oh_train, oh_val, oh_test, _ = make_split_datasets(faithful, one_hot=True)

x1, y1 = next(iter(oh_val))
print('one-hot label batch:', y1.shape, y1.dtype)
print('first 3 rows:\n', np.asarray(y1[:3]))
print('row sums (should all be 1):', np.asarray(tf.reduce_sum(y1, axis=1))[:5])

x2, y2 = next(iter(val_ds))
same = np.array_equal(np.argmax(np.asarray(y1), axis=1), np.asarray(y2).astype(int))
print('argmax(one-hot) == sparse labels:', same)
assert same, 'one-hot and sparse label streams disagree'

## 4. Label alignment for evaluation

`evaluate_utils.predict()` compares model output against labels drawn from the dataset in order. That
is only valid if the val/test datasets are **not shuffled** and drop nothing - so verify it directly
rather than trusting the flag.

**Looks right**: the dataset's label sequence is identical to `frames['test']['label']`, and repeated
iteration yields the same order every time.

In [ ]:
ds_labels = np.concatenate([np.asarray(y) for _, y in test_ds.as_numpy_iterator()])
frame_labels = labels_of(frames['test'])

print('n from dataset:', len(ds_labels), '| n from frame:', len(frame_labels))
print('identical order :', np.array_equal(ds_labels.astype(int), frame_labels))

again = np.concatenate([np.asarray(y) for _, y in test_ds.as_numpy_iterator()])
print('stable across iterations:', np.array_equal(ds_labels, again))
assert np.array_equal(ds_labels.astype(int), frame_labels), 'test labels are out of order'
print('\nalignment checks: PASSED - predictions will line up with y_true')

## 5. Sample images, one row per class

**Looks right**: recognisable CT slices, correct class titles, no all-black or all-white tiles.

In [ ]:
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(10, 2.6 * NUM_CLASSES))
for r, cls in enumerate(CLASS_NAMES):
    sub = frames['train'][frames['train']['class'] == cls].head(4)
    for c in range(4):
        ax = axes[r, c]
        ax.axis('off')
        if c < len(sub):
            img = plt.imread(sub.iloc[c]['filepath'])
            ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
            if c == 0:
                ax.set_title(cls, loc='left', fontsize=10)
fig.suptitle('Training samples by class (raw files, before resize)', y=1.0)
fig.tight_layout()
out = FIGURES_DIR / 'preprocessing_samples.png'
fig.savefig(out, dpi=140, bbox_inches='tight')
plt.show()
print('saved', out)

## 6. Augmentation preview

The augmenter is deliberately conservative: horizontal flip, +/-5% rotation, 10% zoom, 5%
translation. CT slices have a fixed anatomical orientation, so vertical flips and large rotations
would be teaching the model something untrue about the data.

**Looks right**: the same slice, visibly but mildly perturbed; anatomy still upright and recognisable.

In [ ]:
from src.data_utils import get_augmenter

augmenter = get_augmenter(seed=SEED)
base = x[:1]

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(np.asarray(base[0]).astype('uint8'))
axes[0].set_title('original')
axes[0].axis('off')
for i in range(1, 6):
    aug = augmenter(base, training=True)
    axes[i].imshow(np.asarray(aug[0]).astype('uint8'))
    axes[i].set_title(f'augmented {i}')
    axes[i].axis('off')
fig.tight_layout()
out = FIGURES_DIR / 'preprocessing_augmentation.png'
fig.savefig(out, dpi=140, bbox_inches='tight')
plt.show()
print('saved', out)
print('\nAugmentation is applied to the TRAINING dataset only - val/test are never augmented.')
print('\nnext: 02_train_miniconvnet.ipynb')